# CPA on Firmware Implementation of Magma (GOST)

## Magma Trace Capture

In [45]:
SCOPETYPE = 'CWNANO' # options: OPENADC, CWNANO  
PLATFORM = 'CWNANO' # options: CWLITEXMEGA/CW308_XMEGA, CWLITEARM/CW308_STM32F3, CWNANO 
CRYPTO_TARGET='MAGMA'
SS_VER='SS_VER_1_1'

The history saving thread hit an unexpected error (OperationalError('attempt to write a readonly database')).History will not be written to the database.


In [46]:
%run "../Setup_Scripts/Setup_Generic.ipynb"

INFO: Caught exception on reconnecting to target - attempting to reconnect to scope first.
INFO: This is a work-around when USB has died without Python knowing. Ignore errors above this line.


OSError: Could not find ChipWhisperer. Is it connected?

OSError: Could not find ChipWhisperer. Is it connected?

In [47]:
scope.adc.samples = 50000 #options: 96000 for CW-PRO (CW1200), 24400 for CW-Lite, 131070 for CW-Husky

(ChipWhisperer Other ERROR|File util.py:419) Setting unknown attribute samples in <class 'chipwhisperer.capture.scopes.cwnano.ADCSettings'>


AttributeError: 'NoneType' object has no attribute 'controlWrite'

In [48]:
%%bash -s "$PLATFORM" "$CRYPTO_TARGET" "$SS_VER"
cd ../../firmware/mcu/simpleserial-magma
make PLATFORM=$1 CRYPTO_TARGET=$2 SS_VER=$3

Building for platform CWNANO with CRYPTO_TARGET=MAGMA
SS_VER set to SS_VER_1_1
SS_VER set to SS_VER_1_1
Blank crypto options, building for AES128
.
Welcome to another exciting ChipWhisperer target build!!
arm-none-eabi-gcc (15:13.2.rel1-2) 13.2.1 20231009
Copyright (C) 2023 Free Software Foundation, Inc.
This is free software; see the source for copying conditions.  There is NO
warranty; not even for MERCHANTABILITY or FITNESS FOR A PARTICULAR PURPOSE.

Size after:
   text	   data	    bss	    dec	    hex	filename
   5464	     16	   1560	   7040	   1b80	simpleserial-magma-CWNANO.elf
+--------------------------------------------------------
+ Built for platform CWNANO Built-in Target (STM32F030) with:
+ CRYPTO_TARGET = MAGMA
+ CRYPTO_OPTIONS = AES128C
+--------------------------------------------------------


In [49]:
 cw.program_target(scope, prog, "../../firmware/mcu/simpleserial-magma/simpleserial-magma-{}.hex".format(PLATFORM))

AttributeError: 'NoneType' object has no attribute 'controlWrite'

In [50]:
from tqdm.notebook import trange
import numpy as np
import time
from os import urandom

trace_array = []
textin_array = []

text = urandom(8)

N = 100
for i in trange(N, desc='Capturing traces'):
    scope.arm()
    
    target.simpleserial_write('p', text)
    
    ret = scope.capture()
    if ret:
        print("Target timed out!")
        continue
    
    response = target.simpleserial_read('r', 8)
    
    trace_array.append(scope.get_last_trace())
    textin_array.append(text)
    
    text = urandom(8)
    
trace_array = np.array(trace_array)

Capturing traces:   0%|          | 0/100 [00:00<?, ?it/s]

AttributeError: 'NoneType' object has no attribute 'controlWrite'

In [51]:
scope.dis()
target.dis()

In [52]:
assert len(trace_array) == N
print("✔️ OK to continue!")

AssertionError: 

Again, let's quickly plot a trace to make sure everything looks as expected:

In [53]:
%matplotlib notebook
import matplotlib.pylab as plt

plt.figure()
plt.plot(trace_array[0], 'r')
plt.plot(trace_array[N//2], 'g')
plt.plot(trace_array[N-1], 'b')
plt.show()

<IPython.core.display.Javascript object>

IndexError: list index out of range

## Magma Model and Hamming Weight

In [54]:
SBOXES = [[12, 4, 6, 2, 10, 5, 11, 9, 14, 8, 13, 7, 0, 3, 15, 1],
        [6, 8, 2, 3, 9, 10, 5, 12, 1, 14, 4, 7, 11, 13, 0, 15],
        [11, 3, 5, 8, 2, 15, 10, 13, 14, 1, 7, 4, 12, 9, 6, 0],
        [12, 8, 2, 1, 13, 4, 15, 6, 7, 0, 10, 5, 3, 14, 9, 11],
        [7, 15, 5, 10, 8, 1, 6, 13, 0, 9, 3, 14, 11, 4, 2, 12],
        [5, 13, 15, 6, 9, 2, 12, 10, 11, 7, 8, 1, 4, 3, 14, 0],
        [8, 14, 2, 5, 6, 9, 1, 12, 15, 4, 11, 0, 13, 10, 3, 7],
        [1, 7, 14, 13, 0, 5, 8, 3, 4, 15, 10, 6, 9, 12, 11, 2]]

def apply_sbox(s, _in):
    return (
        (s[0][(_in >> 0) & 0x0F] << 0) +
        (s[1][(_in >> 4) & 0x0F] << 4) +
        (s[2][(_in >> 8) & 0x0F] << 8) +
        (s[3][(_in >> 12) & 0x0F] << 12) +
        (s[4][(_in >> 16) & 0x0F] << 16) +
        (s[5][(_in >> 20) & 0x0F] << 20) +
        (s[6][(_in >> 24) & 0x0F] << 24) +
        (s[7][(_in >> 28) & 0x0F] << 28)
    )

def bytes_to_int(inputdata):
    data = bytearray(inputdata)
    return (
        data[7] | data [6] << 8 | data[5] << 16 | data[4] << 24,
        data[3] | data [2] << 8 | data[1] << 16 | data[0] << 24
    )

def modular_add(x, y, mod=2 ** 32):
    res = x + int(y)
    return res if res < mod else res - mod

def shift_left_11(x):
    return ((x << 11) & (2 ** 32 - 1)) | (x >> (32 - 11))

def round(sbox, key, inputdata, byte):
    s = SBOXES[sbox]
    _in = modular_add(inputdata, key)
    sbox_output = apply_sbox(SBOXES, _in)
    return (sbox_output >> (8 * byte)) & 0xFF

def feistel(sbox, key, inputdata, nrounds):
    s = SBOXES[sbox]
    w = bytearray(key)
    x = [
        w[3 + i * 4] |
        w[2 + i * 4] << 8 |
        w[1 + i * 4] << 16 |
        w[0 + i * 4] << 24 for i in range(8)
    ]
    n1, n2 = bytes_to_int(inputdata)
    for i in range(nrounds):
        n1, n2 = shift_left_11(apply_sbox(s, modular_add(n1, x[i]))) ^ n2, n1
    return n1

HW = [bin(n).count("1") for n in range(0, 256)]

## Developing our Correlation Algorithm 

We'll be testing how good our guess is using a measurement called the Pearson correlation coefficient, which measures the linear correlation between two datasets. 

The actual algorithm is as follows for datasets $X$ and $Y$ of length $N$, with means of $\bar{X}$ and $\bar{Y}$, respectively:

$$r = \frac{cov(X, Y)}{\sigma_X \sigma_Y}$$

$cov(X, Y)$ is the covariance of `X` and `Y` and can be calculated as follows:

$$cov(X, Y) = \sum_{n=1}^{N}[(Y_n - \bar{Y})(X_n - \bar{X})]$$

$\sigma_X$ and $\sigma_Y$ are the standard deviation of the two datasets. This value can be calculated with the following equation:

$$\sigma_X = \sqrt{\sum_{n=1}^{N}(X_n - \bar{X})^2}$$

As you can see, the calulation is actually broken down pretty nicely into some smaller chunks that we can implement with some simple functions.

In [55]:
def mean(X):
    return np.sum(X, axis=0)/len(X)

def std_dev(X, X_bar):
    return np.sqrt(np.sum((X-X_bar)**2, axis=0))

def cov(X, X_bar, Y, Y_bar):
    return np.sum((X-X_bar)*(Y-Y_bar), axis=0)

## Correlation Attack Implementaiton

In [56]:
t_bar = np.sum(trace_array, axis=0)/len(trace_array)
o_t = np.sqrt(np.sum((trace_array - t_bar)**2, axis=0))

cparefs = [0] * 32 #put your key byte guess correlations here
bestguess = [0] * 32 #put your key byte guesses here

numt = len(trace_array) #number of traces
nump = np.shape(trace_array)[1] #number of trace points
round_data = np.zeros((numt, 8), dtype=int)

for rnum in range(8):
    bestround = 0
    for tnum_r in range(numt):
        round_data[tnum_r][rnum] = feistel(rnum, bestguess, textin_array[tnum_r], rnum)

    #plt.figur(figsize=(10,6))
    #for i in range(numt):
    #    plt.plot(trace_array[i], label=f'Trace{i + 1}')
    #plt.title(f'Power Traces for Round {rnum + 1}')
    #plt.xlabel('Time')
    #plt.legend()
    #plt.grid()
    #plt.show()
    
    for bnum in range(4):
        cpaoutput = np.zeros(256)
        maxcpa = np.zeros(256)
        correlations = []
        for kguess in range(256):
            bestroundkey = kguess << (bnum * 8) | bestround 
            hws = np.array([HW[round(rnum, bestroundkey, round_data[tnum][rnum], bnum)] for tnum in range (numt)])
            hws_bar = mean(hws)
            o_hws = std_dev(hws, hws_bar)
            correlation = cov(trace_array, t_bar, hws, hws_bar)
            cpaoutput[kguess] = correlation / (o_t * o_hws)
            maxcpa[kguess] = max(abs(cpaoutput[kguess]))
            correlations.append(cpaoutput[kguess])
        bestround = bestround | (np.argmax(maxcpa) << (bnum * 8))
        bestroundkey |= (np.argmax(maxcpa) << (bnum * 8))
        bestguess[((rnum + 1) * 4) - bnum - 1] = np.argmax(maxcpa)
        cparefs[bnum] = max(maxcpa)

        #plt.figure(figsize=(10, 6))
        #plt.plot(correlations, label='Correlation Coefficients')
        #plt.title(f'Correlation Coefficients for')

print("Best Key Guess: ", end="")
for b in bestguess: print("%02x " % b, end="")
print("\n", cparefs)

/tmp/ipykernel_9694/455139145.py:1: RuntimeWarning: invalid value encountered in scalar divide
  t_bar = np.sum(trace_array, axis=0)/len(trace_array)


IndexError: tuple index out of range

With one final check to make sure you've got the correct key:

In [57]:
key = [0x6c,0xec,0xc6,0x7f,0x28,0x7d,0x08,0x3d,
       0xeb,0x87,0x66,0xf0,0x73,0x8b,0x36,0xcf,
       0x16,0x4e,0xd9,0xb2,0x46,0x95,0x10,0x90,
       0x86,0x9d,0x08,0x28,0x5d,0x2e,0x19,0x3b]

for bnum in range(32):
    assert bestguess[bnum] == key[bnum], \
    "Byte {} failed, expected {:02X} got {:02X}".format(bnum, key[bnum], bestguess[bnum])
print("✔️ OK to continue!")

AssertionError: Byte 0 failed, expected 6C got 00